In [31]:
import os, time, pandas as pd
from dataclasses import dataclass, asdict
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- config --------------------
SITE_ROOT = "https://expinterweb.mites.gob.es"
BASE_URL = f"{SITE_ROOT}/regcon/pub/buscadorTextosEstatal"

def make_driver(headless=False):
    opts = Options()
    if headless: opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

@dataclass
class AgreementSpain:
    codigo: str
    denominacion: str
    naturaleza: str
    autoridad_laboral: str
    estado_vigencia: str
    pdf_link: str | None

# -------------------- scraper --------------------
def scrape_spain_links_only(out_dir="spain_data", target_count=50):
    abs_out_dir = os.path.abspath(out_dir)
    if not os.path.exists(abs_out_dir): os.makedirs(abs_out_dir, exist_ok=True)
    
    d = make_driver(headless=False)
    results = []

    try:
        print(f"Loading {BASE_URL}...")
        d.get(BASE_URL)
        time.sleep(2)

        # 1. Apply Search Filters
        WebDriverWait(d, 10).until(EC.presence_of_element_located((By.ID, "texto"))).send_keys("Convenio")
        Select(d.find_element(By.ID, "idNaturaleza")).select_by_visible_text("CONVENIO COLECTIVO")
        d.find_element(By.ID, "_buscar").click()

        # 2. Wait for the table
        WebDriverWait(d, 20).until(EC.presence_of_element_located((By.XPATH, "//table//tr[td]")))
        
        # 3. Fast-scrape the links
        rows = d.find_elements(By.XPATH, "//table//tr[td]")
        print(f"Found {len(rows)} rows. Extracting metadata and links...")

        for i, row in enumerate(rows[:target_count]):
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) < 6: continue
            
            codigo = cols[0].text.strip()
            
            # Extract the URL from the 'Ver' link
            try:
                ver_link_element = cols[-1].find_element(By.TAG_NAME, "a")
                pdf_url = ver_link_element.get_attribute("href")
            except:
                pdf_url = "Link not found"

            ag = AgreementSpain(
                codigo=codigo,
                denominacion=cols[1].text.strip(),
                naturaleza=cols[2].text.strip(),
                autoridad_laboral=cols[3].text.strip(),
                estado_vigencia=cols[4].text.strip(),
                pdf_link=pdf_url
            )
            results.append(ag)
            
            if (i+1) % 10 == 0:
                print(f"   Processed {i+1} rows...")

    finally:
        d.quit()
        if results:
            df = pd.DataFrame([asdict(r) for r in results])
            csv_path = os.path.join(abs_out_dir, "spain_links_inventory.csv")
            df.to_csv(csv_path, index=False, encoding="utf-8")
            print(f"\nSUCCESS: Inventory of {len(results)} links saved to: {csv_path}")
            return df

# RUN (Set target_count to whatever you need)
scrape_spain_links_only(target_count=100)

Loading https://expinterweb.mites.gob.es/regcon/pub/buscadorTextosEstatal...
Found 10 rows. Extracting metadata and links...
   Processed 10 rows...

SUCCESS: Inventory of 10 links saved to: /Users/lukeschreiber/Downloads/RA Work/Collective-Bargaining-Agreements-Web-Scraping/spain_data/spain_links_inventory.csv


,codigo,denominacion,naturaleza,autoridad_laboral,estado_vigencia,pdf_link
0,41101121012021,SOCIEDAD DE FOMENTO AGRICOLA CASTELLONENSE S.A.,CONVENIO COLECTIVO,Sevilla,,https://bopsevilla.dipusevilla.es/export/sites...
1,43002462012001,"ASOCIACION NUCLEAR ASCO-VANDELLOS II, AIE",CONVENIO COLECTIVO,Tarragona,,https://www.diputaciodetarragona.cat/ebop/inde...
2,43002072012000,Consorci d'Aigües de Tarragona,CONVENIO COLECTIVO,Tarragona,,https://www.diputaciodetarragona.cat/ebop/inde...
3,47000892011991,"ARROYO, S.A.",CONVENIO COLECTIVO,Valladolid,,https://bop.sede.diputaciondevalladolid.es/bol...
4,47000582011981,VALLADOLID AUTOMOVIL S.A. (VASA),CONVENIO COLECTIVO,Valladolid,,https://bop.sede.diputaciondevalladolid.es/bol...
5,52100185012019,CONVENIO COLECTIVO SECTORIAL PARA EL TRANSPORT...,CONVENIO COLECTIVO,Melilla,,https://bomemelilla.es/bome/BOME-B-2024-6184
6,33100771012024,ELECTRONIC TRAFIC S.A.,CONVENIO COLECTIVO,Asturias,,https://sede.asturias.es/bopa/2024/06/26/2024-...
7,90002472011982,"HERMANDAD FARMACEUTICA DEL MEDITERRANEO, SOCIE...",CONVENIO COLECTIVO,Estatal,,https://www.boe.es/boe/dias/2024/06/25/pdfs/BO...
8,36002642011995,TRATAMIENTO INDUSTRIAL DE AGUAS SA (TRAINASA),CONVENIO COLECTIVO,Pontevedra,,https://boppo.depo.gal/web/boppo/detalle/-/bop...
9,36104940012017,"DALPHIMETAL ESPAÑA, S.L.U. - CENTRO DE TRABAJO...",CONVENIO COLECTIVO,Pontevedra,,https://boppo.depo.gal/web/boppo/detalle/-/bop...


In [32]:
# import time
# from selenium.webdriver.common.by import By

# d = make_driver(headless=False)
# try:
#     print("Loading main page...")
#     d.get("https://expinterweb.mites.gob.es/regcon/pub/consultaPublicaEstatal")
    
#     print("\n>>> QUICK! Please click 'BUSCAR EN TEXTOS' on the left menu in the Chrome window! <<<")
#     print("You have 15 seconds to click it...")
#     time.sleep(15) 

#     print("\n--- NEW PAGE URL ---")
#     print(d.current_url)

#     print("\n--- ALL TEXT BOXES ---")
#     inputs = d.find_elements(By.XPATH, "//input[@type='text']")
#     for i in inputs:
#         print(f"ID: {i.get_attribute('id')} | Name: {i.get_attribute('name')}")

#     print("\n--- ALL DROPDOWNS (SELECT) ---")
#     selects = d.find_elements(By.TAG_NAME, "select")
#     for s in selects:
#         print(f"ID: {s.get_attribute('id')} | Name: {s.get_attribute('name')}")

#     print("\n--- ALL BUTTONS ---")
#     buttons = d.find_elements(By.XPATH, "//button | //input[@type='submit'] | //input[@type='button']")
#     for b in buttons:
#         text = b.text.strip() if b.text.strip() else b.get_attribute('value')
#         print(f"Tag: {b.tag_name} | ID: {b.get_attribute('id')} | Text/Value: {text}")

# finally:
#     d.quit()